# Continue 6-tap M2-CL on another Google account (Colab T4)

Finishes the 12 fixed-split M2-CL runs. Finished runs are skipped, and an unfinished run continues from its last saved epoch.

**Upload to the top level of My Drive on this account:**
1. `M2CL_code.zip`, from the repo's `colab_upload/` folder. Don't use the older copy in `Downloads`: its checkpoint code is out of date, and the setup cell will stop if you pick it.
2. `PACS.zip`
3. The `m2cl_v2` zip you downloaded from the previous account. If Drive split it into several zips, upload them all. Don't rename them: their names must start with `m2cl_v2`.

**Then:** Runtime > Change runtime type > **T4 GPU**, and run the cells in order. If this session stops too, rerun everything; nothing already saved is lost.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

DRIVE    = '/content/drive/MyDrive'
CODE_ZIP = f'{DRIVE}/M2CL_code.zip'
PACS_ZIP = f'{DRIVE}/PACS.zip'
OUT      = f'{DRIVE}/m2cl_v2'

import os, shutil, zipfile
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
for path in (CODE_ZIP, PACS_ZIP):
    assert os.path.exists(path), f'Not found: {path}. Upload it to the top level of My Drive.'

shutil.rmtree('/content/repo', ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/repo')
os.chdir('/content/repo')
assert 'latest_prev' in open('src/m2cl/engine.py', encoding='utf-8').read(), \
    'This is the old M2CL_code.zip. Upload the one from the colab_upload folder and run this cell again.'
!pip -q install pyyaml tabulate

if not os.path.exists('/content/PACS_raw'):
    print('Extracting PACS...')
    zipfile.ZipFile(PACS_ZIP).extractall('/content/PACS_raw')

# Use the real images: the folder of four domains whose files are not dct_* (docs/AUDIT.md, A0)
def is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
matches = [r for r, d, _ in os.walk('/content/PACS_raw')
           if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and is_rgb(r)]
assert matches, 'No RGB copy of PACS found in PACS.zip.'
DATA = matches[0]
n_images = sum(len(f) for _, _, f in os.walk(DATA))
print('PACS root:', DATA, '| images:', n_images)
assert n_images == 9991, f'Expected 9,991 images, found {n_images}.'
os.makedirs(OUT, exist_ok=True)

free_gb = shutil.disk_usage(DRIVE).free / 1e9
print(f'Free Drive space: {free_gb:.1f} GB')
if free_gb < 5:
    print('WARNING: less than 5 GB free. Empty the Drive trash before training.')

In [ ]:
# 2. Copy the M2-CL progress out of the uploaded zip. Files already here are kept, so rerunning is safe.
import glob, re

found = sorted(glob.glob(f'{DRIVE}/m2cl_v2*.zip'), key=os.path.getmtime, reverse=True)
assert found, 'No zip starting with m2cl_v2 in My Drive. Upload it and run this cell again.'
copied = 0
for path in found:
    with zipfile.ZipFile(path) as z:
        for info in z.infolist():
            m = re.search(r'(?:^|/)runs/(m2cl_6tap_[^/]+)/([^/]+)$', info.filename.replace('\\', '/'))
            if not m or m[2].endswith('.tmp'):
                continue
            target = os.path.join(OUT, 'runs', m[1], m[2])
            if os.path.exists(target):
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target + '.tmp', 'wb') as dst:
                shutil.copyfileobj(src, dst)
            os.replace(target + '.tmp', target)
            copied += 1
print(f'Read {len(found)} zip(s); copied {copied} files.')

In [ ]:
# 3. Where each run stands
import json
import pandas as pd

def run_status(target, seed):
    run_dir = f'{OUT}/runs/m2cl_6tap_{target}_seed{seed}'
    result = os.path.join(run_dir, 'result.json')
    if os.path.exists(result) and json.load(open(result)).get('status') == 'complete':
        return 'finished'
    for name in ('latest.pt', 'latest_prev.pt'):
        try:
            return f"continues at epoch {torch.load(os.path.join(run_dir, name), map_location='cpu', weights_only=False)['epoch'] + 1}"
        except Exception:
            pass
    return 'not started'

status = pd.DataFrame([{'target': t, 'seed': s, 'status': run_status(t, s)}
                       for t in ('art_painting', 'cartoon', 'photo', 'sketch') for s in (0, 1, 2)])
display(status)
n_finished = (status.status == 'finished').sum()
assert n_finished >= 8, f'Only {n_finished} finished runs found; expected at least 8. Training now would repeat finished work, so check the uploaded zip.'
print(f'{n_finished} of 12 finished. Run the next cell to train the rest.')

In [ ]:
# 4. Train the remaining runs: about 1.5 hours if 8 are already finished
!python scripts/train_lodo.py --config configs/m2cl_6tap.yaml --data_root "$DATA" --out_dir "$OUT/runs"

In [ ]:
# 5. Check the results: this should end with 12 of 12
rows = []
for path in sorted(glob.glob(f'{OUT}/runs/m2cl_6tap_*/result.json')):
    r = json.load(open(path))
    if r.get('status') != 'complete':
        continue
    c = r['config']
    best = os.path.join(os.path.dirname(path), 'best.pt')
    assert (c['model'], c['arch_taps'], c['alpha'], c['tau'], c['lr']) == ('m2cl', 6, 0.003, 1.0, 0.0005), f'Wrong settings in {path}'
    assert os.path.exists(best), f'Missing checkpoint: {best}'
    rows.append({'target': c['target'], 'seed': c['seed'], 'test_acc': r['test_acc'], 'best_epoch': r['best_epoch'],
                 'device': r['device'], 'torch': r['torch']})
runs = pd.DataFrame(rows)
print(f'{len(runs)} of 12 runs complete')
if len(runs):
    display(runs.pivot(index='target', columns='seed', values='test_acc').round(2))
    print(runs.groupby('target').test_acc.agg(['mean', 'std']).round(2))
    print('Macro-average of the domain means:', round(runs.groupby('target').test_acc.mean().mean(), 2))
    print('Devices:', runs.device.unique(), '| torch:', runs.torch.unique())

## Next
Download `My Drive/m2cl_v2/runs/` to the laptop and tell Claude. The paired M2 and M2-CL comparison and the weight-distance check use those folders. Note any run that continued in a new session, because the paper's run log records resumed runs.